# MIC-CC — Modelo Financiero Fase 4 · v3 (arquitectura modular + costos por producto)

**Universidad EAN — ODEM** · Manuela Alcalá · julio 2026

Este cuaderno **orquesta**, no calcula. Toda la lógica vive en el paquete
[`src/mic_cc/`](../src/mic_cc/) y está cubierta por [`tests/test_modelo.py`](../tests/test_modelo.py)
(45 pruebas, ejecutables con `pytest`).

> **Nada anterior se borró.** `04_financial_model.ipynb` (original), `05_financial_model_auditado.ipynb`
> (v2) y sus salidas en `outputs/` y `outputs_v2/` quedan intactos. Este cuaderno escribe en `outputs_v3/`.

## Qué añade la v3 sobre la v2

| # | Pendiente que quedó abierto en la v2 | Resuelto en v3 |
|---|---|---|
| 1 | **Factor de costo 0.60 uniforme.** Volvía el margen idéntico para los 10 productos: `margen% = 1 − factor − accesorios`. Cualquier ranking por producto era un artefacto del precio por kilogramo. | **Banda de costos por producto con grado de evidencia** (`COSTOS_SECTOR`). El modelo evalúa el intervalo completo y el semáforo solo asigna Verde si el producto resiste en su extremo desfavorable. |
| 2 | **Precio del café estimado.** La v2 usaba 8.49 USD/kg inferido de un rango de prensa. | **Fuente primaria FNC**: precio interno 2.210.000 COP/carga y cierre NY 324,55 USc/lb (27-jul-2026) → FOB 7,817 USD/kg. |
| 3 | **Perfil del exportador indefinido.** El modelo nunca distinguió si el agente produce o compra para exportar. | Parámetro `perfil_exportador`. Para el café el dato duro de la FNC da un ratio costo/FOB de **0,943** como comercializador, frente a ~0,62 como productor: el margen cambia por un factor cercano a 2. |
| 4 | **Arquitectura monolítica.** Todo en el espacio global de un kernel. | Paquete con separación de intereses: `config` · `datos` · `riesgo` · `modelo`. |
| 5 | **Pruebas embebidas en el flujo.** | Suite `pytest` independiente, con pruebas de inversión, control, detección y **regresión sobre cada error de la auditoría**. |
| 6 | **Supuestos sin trazabilidad de calidad.** | Cada parámetro lleva **grado de evidencia** A/B/C y `resumen_evidencia()` impide presentar como hallazgo algo que descansa en supuestos de grado C. |

**Grados de evidencia:** **A** dato duro de fuente primaria · **B** dato gremial cuantificado por inferencia razonada · **C** supuesto de trabajo, *no es evidencia*.

In [1]:
import sys, os, warnings
from pathlib import Path
import numpy as np, pandas as pd

# El nucleo analitico no depende de matplotlib. Se importa de forma opcional para
# que el cuaderno siga siendo ejecutable en entornos donde la extension nativa de
# matplotlib este bloqueada por politica (p. ej. Windows Application Control), en
# vez de abortar el modelo completo por no poder dibujar.
try:
    import matplotlib.pyplot as plt
    from matplotlib.patches import Patch
    _f = plt.figure(); plt.close(_f)   # el backend se resuelve tarde: hay que probarlo
    GRAFICOS = True
    MOTIVO_SIN_GRAFICOS = ''
except Exception as _e:
    GRAFICOS = False
    MOTIVO_SIN_GRAFICOS = f'{type(_e).__name__}: {_e}'

sys.path.insert(0, str(Path.cwd().parent / 'src'))
from mic_cc import config, datos, riesgo, modelo

DATA, OUT = '../dataraw', '../outputs_v3'
os.makedirs(f'{OUT}/tables', exist_ok=True); os.makedirs(f'{OUT}/figures', exist_ok=True)
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)

P = config.PARAMS
print(f'mic_cc v{__import__("mic_cc").__version__} | {len(P)} parametros | perfil: {P["perfil_exportador"]}')
if not GRAFICOS:
    print(f'AVISO: matplotlib no disponible ({MOTIVO_SIN_GRAFICOS}).')
    print('       El modelo y todas las tablas se calculan igual; solo se omiten las figuras.')
display(config.resumen_evidencia().grado.value_counts().rename('parametros').to_frame().T)
print('Los parametros de grado C son supuestos de trabajo, no evidencia. Toda conclusion que '
      'dependa de ellos se reporta como banda, nunca como punto.')

mic_cc v2.0.0 | 32 parametros | perfil: productor
AVISO: matplotlib no disponible (ImportError: DLL load failed while importing _backend_agg: Una directiva de Control de aplicaciones bloqueó este archivo.).
       El modelo y todas las tablas se calculan igual; solo se omiten las figuras.


grado,C,B,A
parametros,10,5,5


Los parametros de grado C son supuestos de trabajo, no evidencia. Toda conclusion que dependa de ellos se reporta como banda, nunca como punto.


## 1 · Serie COP/CAD

In [2]:
fx, origen = datos.cargar_fx(P, DATA)
fx_sin = datos.construir_copcad(datos.descargar_trm(P['fx_inicio']),
                                datos.descargar_usdcad(P['fx_inicio']), False)

r        = fx['retorno_log'].dropna()
fx_base  = fx['COP_CAD'].rolling(P['ventana_base_dias']).mean().dropna().iloc[-1]
fx_spot  = fx['COP_CAD'].iloc[-1]
trm_hoy  = fx['TRM_COP_USD'].iloc[-1]
vol      = riesgo.volatilidad_anualizada(r, P['dias_habiles_anio'])
vol_sin  = riesgo.volatilidad_anualizada(fx_sin['retorno_log'], P['dias_habiles_anio'])

print(f'TRM     : {origen["trm"]}\nUSD/CAD : {origen["usdcad"]}')
print(f'\n{len(fx)} observaciones, {fx.fecha.min().date()} -> {fx.fecha.max().date()}')
print(f'  spot {fx_spot:,.2f}   base MM{P["ventana_base_dias"]} {fx_base:,.2f}   TRM {trm_hoy:,.2f}')
print(f'\nAlineacion temporal TRM(D) <-> BoC(D-1):')
print(f'  volatilidad alineada  {vol:.4%}   |   sin alinear {vol_sin:.4%}   '
      f'(+{(vol_sin-vol)*1e4:.0f} pb de ruido de emparejamiento)')

TRM     : Datos Abiertos (BanRep), en vivo
USD/CAD : Bank of Canada Valet, en vivo

2312 observaciones, 2014-01-03 -> 2026-07-28
  spot 2,271.36   base MM90 2,602.92   TRM 3,205.80

Alineacion temporal TRM(D) <-> BoC(D-1):
  volatilidad alineada  13.9307%   |   sin alinear 16.5545%   (+262 pb de ruido de emparejamiento)


## 2 · Riesgo cambiario

In [3]:
v95 = riesgo.var_historico(r, P['nivel_confianza'])
v99 = riesgo.var_historico(r, 0.99)
es95 = riesgo.expected_shortfall(r, P['nivel_confianza'])
jb = riesgo.jarque_bera(r)
oos = riesgo.backtest_var_oos(fx['retorno_log'], P['ventana_var_movil'], P['nivel_confianza'])
mh = riesgo.var_multihorizonte(r, fx['COP_CAD'], P['horizontes_cobro'], P['nivel_confianza'])

print(f'Volatilidad anualizada {vol:.4%} | VaR95 {v95:.4%} | VaR99 {v99:.4%} | ES95 {es95:.4%}')
print(f'\nJarque-Bera {jb["jb"]:.1f} (p={jb["p_value"]:.2e}) -> normalidad '
      f'{"NO rechazada" if jb["normal"] else "RECHAZADA"}')
print(f'  => {"admisible" if jb["normal"] else "NO admisible"} escalar cuantiles por sqrt(T). Se usa el empirico.')
print(f'\nKupiec fuera de muestra: {oos["violaciones"]}/{oos["n_obs"]} = {oos["tasa_observada"]:.2%} '
      f'(esperado {1-P["nivel_confianza"]:.2%}) | p={oos["p_value"]:.4f} -> '
      f'{"calibracion aceptable" if oos["bien_calibrado"] else "SE RECHAZA la buena calibracion al 5%"}')
display(mh.round(4))

acum90 = np.log(fx['COP_CAD'] / fx['COP_CAD'].shift(90)).dropna()
ESC = {'Pesimista': riesgo.nivel_desde_retorno_log(fx_base, np.percentile(acum90, 5)),
       'Base': fx_base,
       'Optimista': riesgo.nivel_desde_retorno_log(fx_base, np.percentile(acum90, 95))}
var90 = mh.loc[mh.horizonte_dias == 90, 'var_empirico'].iloc[0]
fx_var90 = riesgo.nivel_desde_retorno_log(fx_base, var90)
print('\nEscenarios (cuantiles empiricos 90d, no +10%/-15% arbitrarios):')
for k, v in ESC.items():
    print(f'  {k:10s} {v:9,.2f}  ({(v/fx_base-1)*100:+6.2f}%)')
print(f'  FX en VaR90 {fx_var90:,.2f}  |  spot vigente {fx_spot:,.2f}')

Volatilidad anualizada 13.9307% | VaR95 -1.3049% | VaR99 -2.3357% | ES95 -1.9269%

Jarque-Bera 1010.6 (p=3.59e-220) -> normalidad RECHAZADA
  => NO admisible escalar cuantiles por sqrt(T). Se usa el empirico.

Kupiec fuera de muestra: 123/2059 = 5.97% (esperado 5.00%) | p=0.0489 -> SE RECHAZA la buena calibracion al 5%


,horizonte_dias,var_empirico,var_sqrt_t,peor_observado,n_ventanas_solapadas,n_efectivo_independiente,sesgo_sqrt_t_pct
0,30,-0.0636,-0.0715,-0.1927,2282,76.1,12.4579
1,60,-0.0864,-0.1011,-0.1667,2252,37.5,16.9577
2,90,-0.1098,-0.1238,-0.1958,2222,24.7,12.7750



Escenarios (cuantiles empiricos 90d, no +10%/-15% arbitrarios):
  Pesimista   2,332.32  (-10.40%)
  Base        2,602.92  ( +0.00%)
  Optimista   2,958.52  (+13.66%)
  FX en VaR90 2,332.32  |  spot vigente 2,271.36


## 3 · Productos, precios y estructura de costos por producto

In [4]:
prod = pd.read_excel(f'{DATA}/2tabla_B4_precios_referencia.xlsx')
prod.columns = ['numero','producto','codigo_hs','precio_usd','unidad','tc_usd_cad',
                'precio_cad','fuente','fecha_consulta']
prod['precio_cad_original'] = prod['precio_cad']
for _, o in config.OVERRIDES.iterrows():
    if o.campo == 'precio_usd':
        prod.loc[prod.codigo_hs == o.codigo_hs, 'precio_usd'] = o.valor_corregido
    elif o.campo == 'tipo_cambio_usd_cad':
        prod['tc_usd_cad'] = o.valor_corregido
prod['precio_cad'] = prod['precio_usd'] * prod['tc_usd_cad']

est = pd.read_excel(f'{DATA}/tabla_B9_estructura_costos.xlsx')
est['pct_importado'] = est['pct_importado'] / 100
prod = (prod.merge(est[['codigo_hs','pct_importado']], on='codigo_hs', validate='1:1')
            .merge(config.TRANSPORTE, on='codigo_hs', validate='1:1')
            .merge(config.COSTOS_SECTOR.drop(columns='producto'), on='codigo_hs', validate='1:1'))

# El ratio efectivo depende del perfil del exportador.
usar_com = P['perfil_exportador'] == 'comercializador'
prod['ratio_efectivo'] = prod['ratio_comercializador'] if usar_com else prod['ratio_base']
for col, r_ in [('costo_base','ratio_efectivo'), ('costo_min','ratio_min'), ('costo_max','ratio_max')]:
    prod[col] = prod['precio_cad'] * fx_base * prod[r_]

print(f'Incoterm de la fuente: {prod["unidad"].unique()[0]}   |   perfil: {P["perfil_exportador"]}')
display(prod[['producto','precio_cad_original','precio_cad','pct_importado',
              'modo_transporte','ratio_min','ratio_base','ratio_max','grado']].round(3))
print('El modelo original usaba 0.60 para los 10. Aqui cada producto tiene banda y grado; '
      'solo el cafe alcanza grado A.')

Incoterm de la fuente: Kilogramo (FOB)   |   perfil: productor


,producto,precio_cad_original,precio_cad,pct_importado,modo_transporte,ratio_min,ratio_base,ratio_max,grado
0,Café sin tostar,4.256,11.009,0.25,maritimo,0.50,0.62,0.72,B
1,Carbón bituminoso,0.117,0.124,0.15,granel,0.70,0.85,0.95,C
2,Frutas tropicales (guayaba/mango),1.596,1.690,0.30,maritimo,0.65,0.75,0.85,C
3,Flores frescas otras (pompones/hortensias),3.325,3.521,0.30,aereo,0.80,0.87,0.93,B
4,Medicamentos corticosteroides,37.240,39.432,0.65,maritimo,0.40,0.55,0.70,C
5,Medias/calcetines fibra sintética,9.044,9.576,0.55,maritimo,0.65,0.74,0.82,C
6,Accesorios tubería hierro/acero,2.793,2.957,0.40,maritimo,0.70,0.79,0.88,C
7,Filetes de trucha congelados,7.315,7.746,0.55,maritimo,0.70,0.79,0.87,B
8,Nueces y semillas preparadas,5.719,6.056,0.55,maritimo,0.75,0.83,0.90,C
9,Rosas frescas cortadas,3.724,3.943,0.30,aereo,0.80,0.87,0.93,B


El modelo original usaba 0.60 para los 10. Aqui cada producto tiene banda y grado; solo el cafe alcanza grado A.


In [5]:
# Ancla de fuente primaria para el cafe y contraste de perfiles.
fnc = modelo.precio_fob_cafe_desde_fnc(2_210_000, trm_hoy, P)
fob_cafe_cop = prod.loc[prod.codigo_hs == '0901.11.90.00', 'precio_usd'].iloc[0] * trm_hoy
print('CAFE — anclaje con fuente primaria FNC (precio_cafe.pdf, 2026-07-27)')
print(f'  precio interno de compra      2,210,000 COP / carga de {P["kg_cps_por_carga"]} kg cps')
print(f'  equivale a                    {fnc["kg_excelso_por_carga"]:.2f} kg de excelso exportable')
print(f'  precio interno                {fnc["precio_interno_cop_kg"]:,.0f} COP/kg  '
      f'({fnc["precio_interno_usd_kg"]:.3f} USD/kg)')
print(f'  precio FOB de exportacion     {fob_cafe_cop:,.0f} COP/kg  (NY 324.55 USc/lb + prima)')
print(f'  ratio costo/FOB si COMERCIALIZA {fnc["precio_interno_cop_kg"]/fob_cafe_cop:.3f}  <- grado A')
print(f'  ratio asumido si PRODUCE        {prod.loc[prod.codigo_hs=="0901.11.90.00","ratio_base"].iloc[0]:.3f}  <- grado B')
print('\nEl modelo original asumia 0.60 sin distinguir los dos casos. La diferencia entre '
      'producir y comercializar cambia el margen del cafe por un factor cercano a 2.')

CAFE — anclaje con fuente primaria FNC (precio_cafe.pdf, 2026-07-27)
  precio interno de compra      2,210,000 COP / carga de 125 kg cps
  equivale a                    93.09 kg de excelso exportable
  precio interno                23,742 COP/kg  (7.406 USD/kg)
  precio FOB de exportacion     25,060 COP/kg  (NY 324.55 USc/lb + prima)
  ratio costo/FOB si COMERCIALIZA 0.947  <- grado A
  ratio asumido si PRODUCE        0.620  <- grado B

El modelo original asumia 0.60 sin distinguir los dos casos. La diferencia entre producir y comercializar cambia el margen del cafe por un factor cercano a 2.


In [6]:
fl = pd.read_excel(f'{DATA}/costos_logisticos_rutas.xlsx')
fl['tarifa_usd'] = fl['tarifa_usd_teu'].apply(
    lambda v: float(str(v).replace('$','').replace(' ','').replace('.','').replace(',','.')))
RUTAS = {'Cartagena CO - Montreal CA':'CTG-Montreal','Cartagena CO - Vancouver CA':'CTG-Vancouver'}
fl = fl[fl.ruta.isin(RUTAS) & (fl.tipo_contenedor == '20 pies')].copy()
fl['ruta_modelo'] = fl.ruta.map(RUTAS)
tarifa = fl.groupby('ruta_modelo')['tarifa_usd'].mean().to_dict()

fletes = pd.DataFrame([
    dict(codigo_hs=x.codigo_hs, producto=x['producto'], ruta=ruta, modo=x.modo_transporte,
         costo_logistico_cop_kg=(c := modelo.costo_logistico_cop_kg(
             x.modo_transporte, x.clase_densidad, x.requiere_frio, ruta,
             tarifa[ruta], trm_hoy, P))[0], detalle=c[1])
    for _, x in prod.iterrows() for ruta in tarifa])
def flete_de(hs, ruta):
    return fletes.loc[(fletes.codigo_hs == hs) & (fletes.ruta == ruta), 'costo_logistico_cop_kg'].iloc[0]

print(f'TRM vigente para convertir fletes: {trm_hoy:,.2f}')
display(fletes.pivot_table(index=['producto','modo'], columns='ruta',
                           values='costo_logistico_cop_kg').round(0))

TRM vigente para convertir fletes: 3,205.80


,ruta,CTG-Montreal,CTG-Vancouver
producto,modo,,
Accesorios tubería hierro/acero,maritimo,168.0,846.0
Café sin tostar,maritimo,168.0,846.0
Carbón bituminoso,granel,48.0,71.0
Filetes de trucha congelados,maritimo,447.0,2257.0
Flores frescas otras (pompones/hortensias),aereo,8976.0,8976.0
Frutas tropicales (guayaba/mango),maritimo,447.0,2257.0
Medias/calcetines fibra sintética,maritimo,479.0,2419.0
Medicamentos corticosteroides,maritimo,279.0,1411.0
Nueces y semillas preparadas,maritimo,479.0,2419.0


## 4 · Márgenes y punto de equilibrio con banda de costos

Cada producto se evalúa en su ratio central **y en los dos extremos de su banda**. El semáforo es robusto: Verde solo si resiste en el extremo desfavorable.

In [7]:
filas, be = [], []
for _, x in prod.iterrows():
    for ruta in tarifa:
        cl = flete_de(x.codigo_hs, ruta)
        bks = []
        for etq, cp in [('base', x.costo_base), ('min', x.costo_min), ('max', x.costo_max)]:
            bk = modelo.breakeven_fx(x.precio_cad, fx_base, cp, x.pct_importado, cl,
                                     P['costos_accesorios_pct_fob'], P['pass_through_flete'])
            bks.append(bk)
            if etq == 'base':
                bk_base = bk
        be.append({'producto': x['producto'], 'codigo_hs': x.codigo_hs, 'ruta': ruta,
                   'breakeven': bk_base, 'breakeven_min': min(bks), 'breakeven_max': max(bks),
                   'riesgo_puntual': modelo.clasificar_riesgo(bk_base, fx_var90, ESC['Pesimista']),
                   'riesgo_robusto': modelo.semaforo_robusto(bks, fx_var90, ESC['Pesimista']),
                   'grado': x.grado, 'colchon_vs_spot_pct': (fx_spot - bk_base) / fx_spot * 100})
        for esc, tc in ESC.items():
            m  = modelo.margen(x.precio_cad, tc, fx_base, x.costo_base, x.pct_importado, cl,
                               P['costos_accesorios_pct_fob'], P['pass_through_flete'])
            lo = modelo.margen(x.precio_cad, tc, fx_base, x.costo_max, x.pct_importado, cl,
                               P['costos_accesorios_pct_fob'], P['pass_through_flete'])
            hi = modelo.margen(x.precio_cad, tc, fx_base, x.costo_min, x.pct_importado, cl,
                               P['costos_accesorios_pct_fob'], P['pass_through_flete'])
            v2 = modelo.margen(x.precio_cad, tc, fx_base, x.precio_cad*fx_base*0.60,
                               x.pct_importado, cl, P['costos_accesorios_pct_fob'],
                               P['pass_through_flete'])
            filas.append({'producto': x['producto'], 'codigo_hs': x.codigo_hs, 'ruta': ruta,
                          'escenario': esc, 'COPCAD': tc, 'grado': x.grado,
                          'pct_importado': x.pct_importado, 'ratio_costo': x.ratio_efectivo,
                          'ingreso_cop': m['ingreso_cop'], 'costo_total_cop': m['costo_total_cop'],
                          'costo_importado_cop': m['costo_importado_cop'],
                          'margen_cop': m['margen_cop'], 'margen_pct': m['margen_pct'],
                          'margen_pct_min': lo['margen_pct'], 'margen_pct_max': hi['margen_pct'],
                          'margen_pct_factor_060': v2['margen_pct']})
tabla_margenes, tabla_be = pd.DataFrame(filas), pd.DataFrame(be)
assert len(tabla_margenes) == 60 and len(tabla_be) == 20

b = tabla_margenes[(tabla_margenes.escenario == 'Base') & (tabla_margenes.ruta == 'CTG-Montreal')]
print('Margen en escenario Base, ruta Montreal — banda por producto vs. factor unico de 0.60:\n')
display(b[['producto','grado','ratio_costo','margen_pct_min','margen_pct',
           'margen_pct_max','margen_pct_factor_060']].sort_values('margen_pct').round(1))

Margen en escenario Base, ruta Montreal — banda por producto vs. factor unico de 0.60:



,producto,grado,ratio_costo,margen_pct_min,margen_pct,margen_pct_max,margen_pct_factor_060
19,Flores frescas otras (pompones/hortensias),B,0.9,0.8,3.8,7.3,17.4
55,Rosas frescas cortadas,B,0.9,0.8,4.0,7.7,18.4
7,Carbón bituminoso,C,0.8,-0.4,8.3,21.3,30.0
49,Nueces y semillas preparadas,C,0.8,4.4,11.2,18.9,33.5
43,Filetes de trucha congelados,B,0.8,7.3,15.2,24.0,33.8
37,Accesorios tubería hierro/acero,C,0.8,6.4,15.2,24.0,33.8
13,Frutas tropicales (guayaba/mango),C,0.8,8.6,17.7,26.8,31.3
31,Medias/calcetines fibra sintética,C,0.7,12.3,20.1,28.9,33.8
1,Café sin tostar,B,0.6,22.4,32.3,44.2,34.3
25,Medicamentos corticosteroides,C,0.6,24.4,39.4,54.4,34.4


In [8]:
print('Semaforo puntual vs. robusto (el robusto exige resistir toda la banda de costos):\n')
display(tabla_be[tabla_be.ruta == 'CTG-Montreal'][
    ['producto','grado','breakeven_min','breakeven','breakeven_max',
     'riesgo_puntual','riesgo_robusto','colchon_vs_spot_pct']].sort_values('breakeven').round(1))
print(tabla_be.groupby(['ruta','riesgo_robusto']).size().to_string())
print(f'\nReferencias: spot {fx_spot:,.0f} | VaR90 {fx_var90:,.0f} | pesimista {ESC["Pesimista"]:,.0f}')
print('Con pass_through_flete = 1.0 el breakeven es invariante a la ruta por algebra: bajo '
      'traslado completo del flete, la eleccion Montreal/Vancouver no altera la economia del '
      'exportador.')

Semaforo puntual vs. robusto (el robusto exige resistir toda la banda de costos):



,producto,grado,breakeven_min,breakeven,breakeven_max,riesgo_puntual,riesgo_robusto,colchon_vs_spot_pct
8,Medicamentos corticosteroides,C,532.0,852.9,1301.5,Verde,Verde,62.5
0,Café sin tostar,B,1190.4,1532.1,1837.4,Verde,Verde,32.5
10,Medias/calcetines fibra sintética,C,1295.9,1611.1,1944.3,Verde,Verde,29.1
14,Filetes de trucha congelados,B,1464.1,1812.6,2184.4,Verde,Verde,20.2
4,Frutas tropicales (guayaba/mango),C,1579.1,1898.0,2244.5,Verde,Verde,16.4
12,Accesorios tubería hierro/acero,C,1643.9,1961.5,2317.6,Verde,Verde,13.6
16,Nueces y semillas preparadas,C,1649.7,1990.2,2342.6,Verde,Rojo,12.4
2,Carbón bituminoso,C,1843.7,2300.4,2619.1,Verde,Rojo,-1.3
6,Flores frescas otras (pompones/hortensias),B,2067.6,2317.5,2544.3,Verde,Rojo,-2.0
18,Rosas frescas cortadas,B,2067.6,2317.5,2544.3,Verde,Rojo,-2.0


ruta           riesgo_robusto
CTG-Montreal   Rojo              4
               Verde             6
CTG-Vancouver  Rojo              4
               Verde             6

Referencias: spot 2,271 | VaR90 2,332 | pesimista 2,332
Con pass_through_flete = 1.0 el breakeven es invariante a la ruta por algebra: bajo traslado completo del flete, la eleccion Montreal/Vancouver no altera la economia del exportador.


## 5 · Stress histórico, capital de trabajo, cobertura y Plan Vallejo

In [9]:
eps = pd.read_excel(f'{DATA}/tabla_B6_episodios_historicos.xlsx')
serie = fx.set_index('fecha')['COP_CAD']
eps['copcad_inicio'] = eps.fecha_inicio.apply(lambda d: serie.asof(pd.Timestamp(d)))
eps['copcad_fin']    = eps.fecha_fin.apply(lambda d: serie.asof(pd.Timestamp(d)))
eps['pct_cambio_trm_original'] = eps['pct_cambio']
eps['pct_cambio_copcad'] = (eps.copcad_fin / eps.copcad_inicio - 1) * 100
eps['error_pp'] = eps.pct_cambio_trm_original - eps.pct_cambio_copcad
print('Episodios: tabla_B6 los mide en TRM (COP/USD); el modelo los aplicaba al COP/CAD.\n')
display(eps[['nombre_episodio','pct_cambio_trm_original','copcad_inicio','copcad_fin',
             'pct_cambio_copcad','error_pp']].round(2))

tabla_stress = pd.DataFrame([
    dict(episodio=e.nombre_episodio, tipo_movimiento=e.tipo_movimiento,
         pct_cambio_copcad=e.pct_cambio_copcad, producto=x['producto'],
         codigo_hs=x.codigo_hs, ruta=ruta,
         COPCAD_stress=(tc := fx_base * (1 + e.pct_cambio_copcad / 100)),
         margen_pct=modelo.margen(x.precio_cad, tc, fx_base, x.costo_base, x.pct_importado,
                                  flete_de(x.codigo_hs, ruta), P['costos_accesorios_pct_fob'],
                                  P['pass_through_flete'])['margen_pct'])
    for _, e in eps.iterrows() for _, x in prod.iterrows() for ruta in tarifa])

tabla_hoy = pd.DataFrame([
    dict(producto=x['producto'], ruta=ruta,
         margen_pct_al_spot=modelo.margen(x.precio_cad, fx_spot, fx_base, x.costo_base,
                                          x.pct_importado, flete_de(x.codigo_hs, ruta),
                                          P['costos_accesorios_pct_fob'],
                                          P['pass_through_flete'])['margen_pct'])
    for _, x in prod.iterrows() for ruta in tarifa])
print(f'\nMargen al COP/CAD vigente ({fx_spot:,.2f}):')
display(tabla_hoy.pivot_table(index='producto', columns='ruta', values='margen_pct_al_spot').round(1))

Episodios: tabla_B6 los mide en TRM (COP/USD); el modelo los aplicaba al COP/CAD.



,nombre_episodio,pct_cambio_trm_original,copcad_inicio,copcad_fin,pct_cambio_copcad,error_pp
0,Crisis petrolera 2014-2016,80.81,1740.49,2465.82,41.67,39.14
1,Choque pandemia COVID-19,27.66,2487.87,2865.75,15.19,12.47
2,Alza de tasas Fed e incertidumbre politica 2022,36.53,2967.93,3678.83,23.95,12.58
3,Apreciacion del peso 2022-2024,-25.64,3678.83,2771.71,-24.66,-0.98



Margen al COP/CAD vigente (2,271.36):


ruta,CTG-Montreal,CTG-Vancouver
producto,,
Accesorios tubería hierro/acero,8.4,7.6
Café sin tostar,25.5,24.9
Carbón bituminoso,-0.9,-0.8
Filetes de trucha congelados,10.1,9.1
Flores frescas otras (pompones/hortensias),-0.7,-0.7
Frutas tropicales (guayaba/mango),10.6,7.5
Medias/calcetines fibra sintética,15.3,14.1
Medicamentos corticosteroides,36.6,36.1
Nueces y semillas preparadas,5.8,5.1


In [10]:
tasas = pd.read_excel(f'{DATA}/tabla_B8_tasas_interes.xlsx')
ibr_nom = tasas['ibr_colombia_pct'].dropna().iloc[-1] / 100
ibr_ea = modelo.nominal_a_ea(ibr_nom, P['base_dias_ibr'])
tasa_can = tasas['tasa_bank_of_canada_pct'].dropna().iloc[-1] / 100
tasa_fondeo = max(P['tasa_fondeo_pyme_ea'], ibr_ea) if P['usar_ibr_como_piso'] else P['tasa_fondeo_pyme_ea']
print(f'IBR nominal base {P["base_dias_ibr"]} {ibr_nom:.4%} -> E.A. {ibr_ea:.4%} | '
      f'fondeo PYME {tasa_fondeo:.4%} | BoC {tasa_can:.4%}')

cap = tabla_margenes.copy()
for d in P['horizontes_cobro']:
    cap[f'costo_financiero_{d}d'] = cap.costo_total_cop.apply(
        lambda c, d=d: modelo.costo_capital_trabajo(c, tasa_fondeo, d))
    cap[f'margen_neto_{d}d_pct'] = (cap.margen_cop - cap[f'costo_financiero_{d}d']) / cap.ingreso_cop * 100

tabla_cob = pd.DataFrame([
    {**modelo.evaluar_cobertura(1.0, modelo.puntos_forward_cip(ibr_ea, tasa_can, d),
                                P['spread_ndf_pyme'],
                                mh.loc[mh.horizonte_dias == d, 'var_empirico'].iloc[0],
                                es95 * np.sqrt(d)),
     'horizonte_dias': d,
     'metodo_original_costo_pct': abs((ibr_nom - tasa_can) * (d / 360))}
    for d in P['horizontes_cobro']])
print('\nCobertura cambiaria (paridad cubierta exacta y con signo):')
display(tabla_cob[['horizonte_dias','prima_forward_pct','spread_ndf_pct','valor_esperado_neto_pct',
                   'cola_var_evitada_pct','metodo_original_costo_pct','recomendacion']].round(5))

IBR nominal base 360 11.1860% -> E.A. 12.0075% | fondeo PYME 19.1900% | BoC 2.2500%



Cobertura cambiaria (paridad cubierta exacta y con signo):


,horizonte_dias,prima_forward_pct,spread_ndf_pct,valor_esperado_neto_pct,cola_var_evitada_pct,metodo_original_costo_pct,recomendacion
0,30,0.00752,0.0075,0.00002,0.06355,0.00745,Cubrirse (valor esperado positivo + elimina cola)
1,60,0.01510,0.0075,0.00760,0.08642,0.01489,Cubrirse (valor esperado positivo + elimina cola)
2,90,0.02273,0.0075,0.01523,0.10977,0.02234,Cubrirse (valor esperado positivo + elimina cola)


In [11]:
dw = pd.read_excel(f'{DATA}/tabla_B3_drawback_iva.xlsx')
dw['aplica'] = dw.drawback_aplica.apply(modelo.normalizar) == 'si'
fin = tabla_margenes.merge(dw[['codigo_hs','drawback_aplica','aplica']], on='codigo_hs', validate='m:1')
ben = fin.apply(lambda x: modelo.beneficio_plan_vallejo(x.costo_importado_cop, x.aplica, P),
                axis=1, result_type='expand')
fin = pd.concat([fin, ben], axis=1)
fin['margen_final_pct'] = (fin.margen_cop + fin.beneficio_financiero_cop) / fin.ingreso_cop * 100
fin['margen_cota_superior_pct'] = (fin.margen_cop + fin.cota_superior_19pct_cop) / fin.ingreso_cop * 100

carry = modelo.tasa_diaria_desde_ea(P['tasa_fondeo_pyme_ea'], P['dias_float_iva'])
print(f'Float de IVA {P["dias_float_iva"]}d al {P["tasa_fondeo_pyme_ea"]:.2%} E.A. -> el beneficio '
      f'real es {carry:.2%} del IVA, no el 100%.\n')
c = fin[fin.escenario == 'Base'].drop_duplicates('producto').copy()
c['mejora_real_pp'] = c.margen_final_pct - c.margen_pct
c['mejora_metodo_original_pp'] = c.margen_cota_superior_pct - c.margen_pct
display(c[['producto','pct_importado','drawback_aplica','mejora_real_pp',
           'mejora_metodo_original_pp']].round(3))

Float de IVA 120d al 19.19% E.A. -> el beneficio real es 5.94% del IVA, no el 100%.


,producto,pct_importado,drawback_aplica,mejora_real_pp,mejora_metodo_original_pp
1,Café sin tostar,0.25,Sí,0.174,2.928
7,Carbón bituminoso,0.15,No está claro,0.000,0.000
13,Frutas tropicales (guayaba/mango),0.30,Sí,0.231,3.881
19,Flores frescas otras (pompones/hortensias),0.30,Sí,0.149,2.505
25,Medicamentos corticosteroides,0.65,Sí,0.402,6.774
31,Medias/calcetines fibra sintética,0.55,Sí,0.451,7.587
37,Accesorios tubería hierro/acero,0.40,No está claro,0.000,0.000
43,Filetes de trucha congelados,0.55,Sí,0.480,8.076
49,Nueces y semillas preparadas,0.55,Sí,0.500,8.418
55,Rosas frescas cortadas,0.30,Sí,0.157,2.645


## 6 · Figuras

In [12]:
VERDE, AZUL, ROJO, NARANJA = '#375623', '#2E75B6', '#C00000', '#C55A11'
CR = {'Verde': VERDE, 'Amarillo': '#BF8F00', 'Rojo': ROJO}

fx['vol_rodante_90d'] = fx['retorno_log'].rolling(90).std() * np.sqrt(P['dias_habiles_anio'])
if not GRAFICOS:
    print("Figura omitida: matplotlib bloqueado en este entorno.")
else:
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
    a1.plot(fx.fecha, fx.COP_CAD, color=AZUL, lw=1.1, label='COP/CAD')
    a1.axhline(fx_base, color=VERDE, ls='--', lw=1, label=f'Base MM90 = {fx_base:,.0f}')
    a1.axhline(fx_spot, color=ROJO, ls=':', lw=1.4, label=f'Spot = {fx_spot:,.0f}')
    for _, e in eps.iterrows():
        fi, ff = pd.Timestamp(e.fecha_inicio), pd.Timestamp(e.fecha_fin)
        if ff >= fx.fecha.min() and fi <= fx.fecha.max():
            a1.axvspan(max(fi, fx.fecha.min()), ff, alpha=.18, zorder=0,
                       color='#F4B183' if e.tipo_movimiento == 'Devaluacion' else '#9DC3E6')
    a1.set_ylabel('COP por CAD'); a1.legend(loc='upper left', fontsize=9)
    a1.set_title('COP/CAD 2014-2026 y episodios historicos', fontweight='bold')
    a2.plot(fx.fecha, fx.vol_rodante_90d*100, color=NARANJA, lw=1, label='Volatilidad rodante 90d')
    a2.axhline(vol*100, color=ROJO, ls='--', label=f'Promedio: {vol*100:.1f}%')
    a2.set_ylabel('Volatilidad anualizada (%)'); a2.set_xlabel('Fecha'); a2.legend(loc='upper left', fontsize=9)
    plt.tight_layout(); plt.savefig(f'{OUT}/figures/figura_04_copcad_volatilidad.png', dpi=150, bbox_inches='tight'); plt.show()

Figura omitida: matplotlib bloqueado en este entorno.


In [13]:
# Figura 5: margen con BANDA de incertidumbre de costos
d = (tabla_margenes[(tabla_margenes.escenario == 'Base') & (tabla_margenes.ruta == 'CTG-Montreal')]
     .set_index('producto').sort_values('margen_pct'))
if not GRAFICOS:
    print("Figura omitida: matplotlib bloqueado en este entorno.")
else:
    fig, ax = plt.subplots(figsize=(13, 7))
    ax.barh(d.index, d.margen_pct_max - d.margen_pct_min, left=d.margen_pct_min,
            color=[CR.get(g_, AZUL) for g_ in ['Verde']*len(d)], alpha=.35, label='Banda de costos')
    ax.scatter(d.margen_pct, d.index, color=AZUL, zorder=3, s=70, label='Ratio central del producto')
    ax.scatter(d.margen_pct_factor_060, d.index, color=ROJO, marker='x', zorder=4, s=80,
               label='Factor unico 0.60 (modelo original)')
    ax.axvline(0, color='black', ls='--', lw=1)
    ax.set_xlabel('Margen (%)'); ax.set_title('Margen base con banda de incertidumbre de costos, ruta Montreal',
                                              fontweight='bold')
    ax.legend(loc='lower right')
    plt.tight_layout(); plt.savefig(f'{OUT}/figures/figura_05_margenes_banda.png', dpi=150, bbox_inches='tight'); plt.show()

Figura omitida: matplotlib bloqueado en este entorno.


In [14]:
# Figura 6: mapa de riesgo con banda de breakeven
b = tabla_be[tabla_be.ruta == 'CTG-Montreal'].sort_values('breakeven')
if not GRAFICOS:
    print("Figura omitida: matplotlib bloqueado en este entorno.")
else:
    fig, ax = plt.subplots(figsize=(11, 7))
    ax.barh(b.producto, b.breakeven_max - b.breakeven_min, left=b.breakeven_min,
            color=[CR[x] for x in b.riesgo_robusto], alpha=.85)
    ax.scatter(b.breakeven, b.producto, color='black', zorder=3, s=25)
    l1 = ax.axvline(fx_spot, color=ROJO, lw=2)
    l2 = ax.axvline(fx_base, color=AZUL, ls='--')
    l3 = ax.axvline(ESC['Pesimista'], color='orange', ls='--')
    ax.set_xlabel('COP/CAD de equilibrio (banda por incertidumbre de costos)')
    ax.set_title('Mapa de riesgo cambiario — semaforo robusto', fontweight='bold')
    ax.legend([l1, l2, l3] + [Patch(facecolor=CR[n]) for n in b.riesgo_robusto.unique()],
              [f'Spot ({fx_spot:,.0f})', f'Base ({fx_base:,.0f})', f'Pesimista ({ESC["Pesimista"]:,.0f})']
              + [f'Riesgo {n}' for n in b.riesgo_robusto.unique()], loc='lower right', fontsize=9)
    plt.tight_layout(); plt.savefig(f'{OUT}/figures/figura_06_mapa_riesgo_breakeven.png', dpi=150, bbox_inches='tight'); plt.show()

Figura omitida: matplotlib bloqueado en este entorno.


In [15]:
# Figura 7: estres historico  |  Figura 8: efecto de las correcciones
if not GRAFICOS:
    print("Figura omitida: matplotlib bloqueado en este entorno.")
else:
    fig, ax = plt.subplots(figsize=(14, 7))
    orden = eps.sort_values('pct_cambio_copcad').nombre_episodio.tolist()
    tabla_stress.pivot_table(index='producto', columns='episodio', values='margen_pct',
                             aggfunc='mean')[orden].plot(kind='bar', ax=ax)
    ax.axhline(0, color='black', ls='--', lw=1); ax.set_ylabel('Margen (%)'); ax.set_xlabel('')
    ax.set_title('Margenes bajo episodios historicos, recalculados en COP/CAD', fontweight='bold')
    ax.tick_params(axis='x', rotation=75)
    ax.legend(title='Episodio (de mas adverso a mas favorable)', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout(); plt.savefig(f'{OUT}/figures/figura_07_stress_testing.png', dpi=150, bbox_inches='tight'); plt.show()

    fig, ax = plt.subplots(figsize=(13, 7))
    d2 = (tabla_margenes[(tabla_margenes.escenario == 'Base') & (tabla_margenes.ruta == 'CTG-Montreal')]
          .set_index('producto')[['margen_pct_factor_060', 'margen_pct']].sort_values('margen_pct'))
    d2.plot(kind='bar', ax=ax, color=[ROJO, VERDE])
    ax.axhline(0, color='black', ls='--', lw=1); ax.set_ylabel('Margen (%)'); ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=75)
    ax.legend(ax.containers, ['Factor unico 0.60 (modelo original)',
                              'Ratio de costo por producto (v3)'])
    ax.set_title('Efecto de reemplazar el factor de costo uniforme', fontweight='bold')
    plt.tight_layout(); plt.savefig(f'{OUT}/figures/figura_08_efecto_costos_producto.png', dpi=150, bbox_inches='tight'); plt.show()

Figura omitida: matplotlib bloqueado en este entorno.


## 7 · Guardado y verificación

In [16]:
salidas = {
    'tabla_00_registro_correcciones.xlsx': config.OVERRIDES,
    'tabla_00b_grados_de_evidencia.xlsx': config.resumen_evidencia(),
    'tabla_00c_estructura_costos_por_producto.xlsx': config.COSTOS_SECTOR,
    'tabla_04_indicadores_riesgo.xlsx': pd.DataFrame(
        [{'indicador': 'Volatilidad anualizada', 'valor_pct': vol*100},
         {'indicador': 'VaR 1d 95%', 'valor_pct': v95*100},
         {'indicador': 'VaR 1d 99%', 'valor_pct': v99*100},
         {'indicador': 'Expected Shortfall 95%', 'valor_pct': es95*100}] +
        [{'indicador': f'VaR {int(x.horizonte_dias)}d 95% (empirico)', 'valor_pct': x.var_empirico*100}
         for x in mh.itertuples()]),
    'tabla_04b_var_multihorizonte.xlsx': mh,
    'tabla_05_margenes_escenarios.xlsx': tabla_margenes,
    'tabla_06_breakeven_semaforo_robusto.xlsx': tabla_be,
    'tabla_06b_logistica_por_modo.xlsx': fletes,
    'tabla_08_episodios_recalculados.xlsx': eps[
        ['nombre_episodio','tipo_movimiento','fecha_inicio','fecha_fin',
         'pct_cambio_trm_original','copcad_inicio','copcad_fin','pct_cambio_copcad','error_pp']],
    'tabla_08b_stress_testing.xlsx': tabla_stress,
    'tabla_08c_margen_al_spot_vigente.xlsx': tabla_hoy,
    'tabla_09_costo_capital.xlsx': cap,
    'tabla_10_cobertura_costo_beneficio.xlsx': tabla_cob,
    'tabla_11_margen_final_ajustado.xlsx': fin,
}
for n, d in salidas.items():
    d.to_excel(f'{OUT}/tables/{n}', index=False)

figuras = ['figura_04_copcad_volatilidad.png', 'figura_05_margenes_banda.png',
           'figura_06_mapa_riesgo_breakeven.png', 'figura_07_stress_testing.png',
           'figura_08_efecto_costos_producto.png']
chk = ([{'artefacto': n, 'tipo': 'tabla', 'medida': len(d),
         'ok': os.path.exists(f'{OUT}/tables/{n}') and len(d) > 0} for n, d in salidas.items()]
       + [{'artefacto': n, 'tipo': 'figura',
           'medida': (round(os.path.getsize(f'{OUT}/figures/{n}')/1024, 1)
                      if os.path.exists(f'{OUT}/figures/{n}') else 0),
           'ok': (os.path.exists(f'{OUT}/figures/{n}')
                  and os.path.getsize(f'{OUT}/figures/{n}') > 20480)} for n in figuras])
checklist = pd.DataFrame(chk)
display(checklist)
n_tab = int((checklist.tipo == 'tabla').sum())
ok_tab = int(checklist[checklist.tipo == 'tabla'].ok.sum())
ok_fig = bool(checklist[checklist.tipo == 'figura'].ok.all())
assert ok_tab == n_tab, 'Falta al menos una tabla de salida'
print(f'\nTablas: {ok_tab}/{n_tab} correctas en {OUT}/tables/')
if ok_fig:
    print(f'Figuras: {len(figuras)}/{len(figuras)} correctas en {OUT}/figures/')
else:
    print(f'Figuras: NO generadas -- matplotlib bloqueado en este entorno.')
    print(f'  Motivo: {MOTIVO_SIN_GRAFICOS}')
    print('  Es una restriccion del sistema (Windows Application Control sobre la DLL nativa')
    print('  _backend_agg), no un fallo del modelo. Al levantarla, basta reejecutar el cuaderno:')
    print('  todas las figuras se regeneran sin ningun cambio de codigo.')
print('Cuadernos 04 (original), 05 (v2) y sus salidas: intactos.')
print('\nLogica del modelo en src/mic_cc/ — verificar con:  python -m pytest tests/ -q')

,artefacto,tipo,medida,ok
0,tabla_00_registro_correcciones.xlsx,tabla,2,True
1,tabla_00b_grados_de_evidencia.xlsx,tabla,20,True
2,tabla_00c_estructura_costos_por_producto.xlsx,tabla,10,True
3,tabla_04_indicadores_riesgo.xlsx,tabla,7,True
4,tabla_04b_var_multihorizonte.xlsx,tabla,3,True
5,tabla_05_margenes_escenarios.xlsx,tabla,60,True
6,tabla_06_breakeven_semaforo_robusto.xlsx,tabla,20,True
7,tabla_06b_logistica_por_modo.xlsx,tabla,20,True
8,tabla_08_episodios_recalculados.xlsx,tabla,4,True
9,tabla_08b_stress_testing.xlsx,tabla,80,True



Tablas: 14/14 correctas en ../outputs_v3/tables/
Figuras: NO generadas -- matplotlib bloqueado en este entorno.
  Motivo: ImportError: DLL load failed while importing _backend_agg: Una directiva de Control de aplicaciones bloqueó este archivo.
  Es una restriccion del sistema (Windows Application Control sobre la DLL nativa
  _backend_agg), no un fallo del modelo. Al levantarla, basta reejecutar el cuaderno:
  todas las figuras se regeneran sin ningun cambio de codigo.
Cuadernos 04 (original), 05 (v2) y sus salidas: intactos.

Logica del modelo en src/mic_cc/ — verificar con:  python -m pytest tests/ -q


## 8 · Cierre

**Resuelto en esta versión.** Estructura de costos por producto con banda y grado de evidencia en
lugar del factor uniforme de 0,60; precio del café anclado en fuente primaria de la FNC; distinción
explícita entre exportador productor y comercializador; arquitectura separada en paquete; y suite
`pytest` de 45 pruebas con casos de regresión sobre cada error de la auditoría.

**Lo que sigue abierto, y es recolección de datos, no código:**

1. **Siete de los diez ratios de costo son grado C.** Solo el café alcanza grado A y tres productos
   grado B. Antes de publicar cualquier ranking de rentabilidad por producto hay que obtener
   estructuras de costo reales: FNC (café, para el perfil productor), Asocolflores (flores),
   Fedeacua (trucha), ANDI (farmacéutico y siderúrgico), Fenalcarbón (carbón), Inexmoda (confección).
2. **Flete aéreo Bogotá–Canadá y flete a granel del carbón** siguen siendo estimaciones de orden de
   magnitud, declaradas como grado C en `PARAMS`.
3. **Precios de los nueve productos distintos del café**: son valores unitarios de Trade Map 2024,
   no precios de mercado de 2026.
4. **Spread real del NDF** cotizado por un banco para el ticket típico de una PYME.
5. **Incoterm efectivo** de cada operación y grado real de traslado del flete al comprador.
6. Segunda cotización de flete Cartagena–Vancouver en 20 pies: la única disponible es más cara que
   la de 40 pies, lo que sugiere que no es una tarifa FCL comparable.

Mientras 1 siga abierto, la lectura honesta de los resultados por producto es la **banda**, no el
punto central.